In [ ]:
import wandb
import pandas as pd
# 初始化 API
api = wandb.Api()

# 设置实体和项目
entity = "wangyuhao"  # 替换为你的WandB实体名
project = "DMTNML"  # 替换为你的WandB项目名

# 获取项目中的所有sweep（通过获取项目中的运行并过滤出sweep）
runs = api.runs(path=f"{entity}/{project}")

# 过滤出与sweep相关的runs，并提取唯一的sweep_id
sweep_ids = set(run.sweep.id for run in runs if run.sweep)


: 

In [ ]:

# 列出所有sweep的ID
targets=["fct","fknn","mrre","mrre_XZ","mrre_ZX","svc","svc_rbf"]
datasets=["ACT","coil20","gast","SAM","mnist","kmnist","HCL","emnist","MCA","coil100"]
methods=["ivis","_tsne","pumap","pacmap","_umap"]
results = {
    target: {dataset: {method: None for method in methods} for dataset in datasets}
    for target in targets
}
for sweep_id in sweep_ids:
    sweep = api.sweep(f"{entity}/{project}/{sweep_id}")
    for dataset in datasets:
        for method in methods:
            sweep_name=method+dataset+"_repeat1"
            if sweep.name == sweep_name:
                print(sweep_name)
                sweep_runs = sweep.runs
                for target in targets:
                    max_train_target = None
                    for run in sweep_runs:
                        train_target = run.summary.get(f'train_{target}', None)
                        if train_target is not None:
                            if max_train_target is None or train_target > max_train_target:
                                max_train_target = train_target
                        
                        # 存储最大值在结果字典中
                        if max_train_target is not None:
                            results[target][dataset][method] = max_train_target
for target in targets:
    df = pd.DataFrame(results[target]).T  # 转置使datasets为行，methods为列
    
    # 保存到Excel文件，并确保行列名称被保存
    file_name = f"{target}_train_max_values.xlsx"
    df.to_excel(file_name, index=True, header=True)
    print(f"Saved {file_name}")
